# Desafío RNN para predecir una señal

Proponé al menos dos arquitecturas recurrentes para predecir la próxima muestra de una señal sintética. Revisá el MSE de entrenamiento y validación de cada intento. Luego podés compartir los resultados que quieras comparar en clase.

El ranking usa el **menor MSE de validación**. El conjunto de test queda reservado para una evaluación final del docente. Compará modelos entrenados durante la misma cantidad de épocas y tené en cuenta cuántos parámetros necesita cada uno.

## 1. Configuración común

Usamos la señal de `RNN_signal.ipynb`, con 50 muestras para predecir un valor. De las 10.000 series, las primeras 7.000 son de entrenamiento y las siguientes 2.000 de validación. La semilla, el tamaño de batch y Adam con `lr=1e-3` son comunes a todos los intentos. Cada entrenamiento completa las épocas indicadas en `EPOCHS` y recupera los pesos de la época con menor MSE de validación.

In [ ]:
import re
import uuid
from getpass import getpass
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

PROTOCOLO = "RNN_SIGNAL_v1"
SEED = 42
N_SERIES = 10_000
N_STEPS = 50
N_FUTURE = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
intentos = []
print(f"Dispositivo: {device}")

## 2. Datos del desafío

In [ ]:
# Generamos la serie temporal sintética del notebook de clase.

def generate_time_series_complex(batch_size, n_steps, n_future=1, include_derivative=False):
    total_steps = n_steps + n_future
    time = np.linspace(0.0, 1.0, total_steps, dtype=np.float32)

    # generamos 3 frecuencias, fases y una envolvente de amplitud que varía lentamente, para cada serie del batch
    freq1 = np.random.uniform(8.0, 14.0, size=(batch_size, 1))
    freq2 = np.random.uniform(16.0, 26.0, size=(batch_size, 1))
    freq3 = np.random.uniform(2.0, 5.0, size=(batch_size, 1))

    phase1 = np.random.uniform(0.0, 2.0 * np.pi, size=(batch_size, 1))
    phase2 = np.random.uniform(0.0, 2.0 * np.pi, size=(batch_size, 1))
    phase3 = np.random.uniform(0.0, 2.0 * np.pi, size=(batch_size, 1))

    amplitude = 0.7 + 0.3 * np.sin(2.0 * np.pi * (time[None, :] * np.random.uniform(0.8, 1.5, size=(batch_size, 1)) + np.random.uniform(0.0, 1.0, size=(batch_size, 1))))

    # se contruyen las 3 series trigonometricas
    base = amplitude * np.sin(freq1 * time[None, :] + phase1)
    harmonic = 0.35 * np.sin(freq2 * time[None, :] + phase2)
    slow_component = 0.2 * np.cos(freq3 * time[None, :] + phase3)

    # se contruye ahora una tendencia lineal y una curvatura cuadrática, con coeficientes aleatorios para cada serie del batch
    slope = np.random.uniform(-0.5, 0.5, size=(batch_size, 1))
    trend = 0.25 * slope * (time[None, :] - 0.5)

    curvature = np.random.uniform(-0.15, 0.15, size=(batch_size, 1))
    quadratic = curvature * (time[None, :] - 0.5) ** 2

    # generamos un ruido
    noise = 0.08 * np.random.randn(batch_size, total_steps)

    # componemos la serie como suma de todo lo anterior
    series = base + harmonic + slow_component + trend + quadratic + noise

    # si se solicita, se calcula la derivada discreta de la serie y se concatena como una segunda feature
    if include_derivative:
        derivative = np.diff(series, axis=1, prepend=series[:, :1])
        series = np.stack([series, derivative], axis=-1)
    else:
        series = series[..., np.newaxis]

    return series.astype(np.float32)

In [ ]:
np.random.seed(SEED)
series = generate_time_series_complex(
    batch_size=N_SERIES, n_steps=N_STEPS, n_future=N_FUTURE, include_derivative=True,
)
X = torch.from_numpy(series[:9_000, :N_STEPS])
y = torch.from_numpy(series[:9_000, N_STEPS:, 0])

def crear_dataloaders(usar_derivada):
    entradas = X if usar_derivada else X[:, :, :1]
    train = TensorDataset(entradas[:7_000], y[:7_000])
    valid = TensorDataset(entradas[7_000:], y[7_000:])
    train_loader = DataLoader(
        train, batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    valid_loader = DataLoader(valid, batch_size=BATCH_SIZE)
    return train_loader, valid_loader

print(f"Entrenamiento: {len(X[:7_000]):,} series · Validación: {len(X[7_000:]):,} series")
print(f"Entrada por serie: {N_STEPS} pasos · Objetivo: {N_FUTURE} muestra futura")

## 3. Modelo de partida

La RNN usa el último estado temporal para predecir la siguiente muestra. Podés modificar la clase o probar distintas configuraciones en **Tu arquitectura**. Si implementás otra red, mantené la interfaz `(batch, 50, canales) → (batch, 1)`. Los canales son el valor de la señal y, si la activás, su diferencia respecto de la muestra anterior.

In [ ]:
# Modelo recurrente base: procesa la secuencia y usa el último estado temporal.

class RNNRegressor(torch.nn.Module):
    def __init__(self, input_size=1, hidden_size=12, num_layers=1, n_out=1, nonlinearity='tanh'):
        super().__init__()
        self.rnn = torch.nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            nonlinearity=nonlinearity,
            batch_first=True,
        )
        self.fc = torch.nn.Linear(hidden_size, n_out)

    def forward(self, x):
        output, _ = self.rnn(x)
        ultimo_estado = output[:, -1, :]
        return self.fc(ultimo_estado)

## 4. Rutina de entrenamiento

Usamos la misma función `train_model` de `RNN_signal.ipynb`. Al terminar, grafica el MSE de entrenamiento y validación por época.

In [ ]:
# Entrenamos con Adam y MSE, y recuperamos el mejor estado según validación.
def train_model(model, dataloaders, model_name, learning_rate=1e-3,
                num_epochs=3, patience=5, device=device):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = torch.nn.MSELoss()
    best_state = None
    best_eval = np.inf
    epochs_without_improvement = 0
    history = {'train_loss': [], 'eval_loss': []}

    for epoch in range(num_epochs):
        model.train()
        train_losses = []
        for X, y in dataloaders['train']:
            X = X.to(device)
            y = y.to(device).flatten(start_dim=1)

            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        eval_losses = []
        with torch.no_grad():
            for X, y in dataloaders['eval']:
                X = X.to(device)
                y = y.to(device).flatten(start_dim=1)
                eval_losses.append(criterion(model(X), y).item())

        train_loss = float(np.mean(train_losses))
        eval_loss = float(np.mean(eval_losses))
        history['train_loss'].append(train_loss)
        history['eval_loss'].append(eval_loss)
        print(f'Época {epoch + 1:2d}/{num_epochs} · MSE train {train_loss:.5f} · MSE validación {eval_loss:.5f}')

        if eval_loss < best_eval:
            best_eval = eval_loss
            epochs_without_improvement = 0
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'Early stopping en la época {epoch + 1}')
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    epochs = range(1, len(history['train_loss']) + 1)
    fig, ax = plt.subplots(figsize=(8, 4), layout='constrained')
    ax.plot(epochs, history['train_loss'], label='Entrenamiento')
    ax.plot(epochs, history['eval_loss'], label='Validación')
    ax.set(title=model_name, xlabel='Época', ylabel='MSE', xticks=list(epochs))
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

    return history

## 5. Tu arquitectura

Cambiá el tamaño del estado oculto, la cantidad de capas, la activación o el uso de derivada. Dale un nombre descriptivo a cada propuesta. Si editaste la clase, volvé a ejecutar también su celda. Cada ejecución completa de la celda de entrenamiento crea un nuevo intento.

In [ ]:
NOMBRE_MODELO = "RNN base"
HIDDEN_SIZE = 12
NUM_LAYERS = 1
NONLINEARITY = "tanh"  # "tanh" o "relu"
USAR_DERIVADA = False
EPOCHS = 20

In [ ]:
if type(EPOCHS) is not int or EPOCHS < 1:
    raise ValueError("EPOCHS debe ser un entero positivo.")
if type(USAR_DERIVADA) is not bool:
    raise ValueError("USAR_DERIVADA debe ser True o False.")
if not NOMBRE_MODELO.strip():
    raise ValueError("Escribí un nombre para el modelo.")

torch.manual_seed(SEED)
train_loader, valid_loader = crear_dataloaders(USAR_DERIVADA)
input_size = train_loader.dataset.tensors[0].shape[-1]
modelo = RNNRegressor(
    input_size=input_size, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
    nonlinearity=NONLINEARITY, n_out=N_FUTURE,
)
n_parametros = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(modelo)
print(f"Parámetros entrenables: {n_parametros:,}")

# Comprobamos la forma de salida antes de entrenar.
entradas_prueba, objetivos_prueba = train_loader.dataset[:2]
modelo.eval()
with torch.no_grad():
    salida_prueba = modelo(entradas_prueba)
if salida_prueba.shape != objetivos_prueba.shape:
    raise ValueError(f"La red devolvió {salida_prueba.shape}; se esperaba {objetivos_prueba.shape}.")

dataloaders = {"train": train_loader, "eval": valid_loader}
inicio = perf_counter()
history = train_model(
    modelo, dataloaders, model_name=f"Intento {len(intentos) + 1} · {NOMBRE_MODELO.strip()}",
    learning_rate=LEARNING_RATE, num_epochs=EPOCHS, patience=EPOCHS,
)
tiempo = perf_counter() - inicio

train_mse = history["train_loss"]
val_mse = history["eval_loss"]
if not np.isfinite([train_mse, val_mse]).all():
    raise ValueError("El MSE dejó de ser finito durante el entrenamiento.")

intento = {
    "run_id": str(uuid.uuid4()),
    "protocolo": PROTOCOLO,
    "modelo": NOMBRE_MODELO.strip(),
    "arquitectura": repr(modelo),
    "usar_derivada": USAR_DERIVADA,
    "n_parametros": n_parametros,
    "epochs": len(val_mse),
    "training_time": tiempo,
    "train_mse": train_mse,
    "val_mse": val_mse,
}
intentos.append(intento)

display(pd.DataFrame([
    {
        "Intento": i, "Modelo": r["modelo"], "Derivada": r["usar_derivada"],
        "Parámetros": r["n_parametros"], "Épocas": r["epochs"],
        "Mejor MSE val.": min(r["val_mse"]), "MSE val. final": r["val_mse"][-1],
    }
    for i, r in enumerate(intentos, start=1)
]).round(6))

## 6. Enviar un intento

Completá el alias, la cohorte y la URL `/exec` que indicó el docente. Elegí qué intento enviar según el número mostrado en la tabla. Con `NUMERO_INTENTO = None` se envía el último. El token se solicita al ejecutar la celda.

Si se corta la conexión, podés reejecutar esta celda para el mismo intento. La hoja usa el identificador de la corrida para evitar duplicados.

In [ ]:
ALUMNO = ""
COHORTE = ""
ENDPOINT = "https://script.google.com/macros/s/AKfycbxaj-QP3j6tZ8jWq0zmtpARovkUrzuxvtkgiY0e5AuTJGcEw2auCl5u-T4IxG9348N-/exec"
NUMERO_INTENTO = None  # None usa el último intento. También podés escribir 1, 2, etc.


def enviar_resultado(intento, alumno, cohorte, endpoint, token):
    payload = {**intento, "alumno": alumno, "cohorte": cohorte, "token": token}
    try:
        respuesta = requests.post(endpoint, json=payload, timeout=30)
        respuesta.raise_for_status()
        datos = respuesta.json()
    except (requests.RequestException, ValueError) as exc:
        print(f"No se pudo enviar el intento: {exc}")
        return

    if not isinstance(datos, dict):
        print("El servidor devolvió una respuesta inesperada.")
    elif datos.get("ok") and datos.get("duplicado"):
        print(f"Este intento ya estaba registrado: {datos['run_id']}")
    elif datos.get("ok"):
        print(f"Intento registrado: {datos['run_id']}")
    else:
        print(f"El servidor rechazó el envío: {datos.get('error', 'error desconocido')}")


numero = len(intentos) if NUMERO_INTENTO is None else NUMERO_INTENTO
if not intentos:
    print("Entrená un modelo antes de enviar.")
elif type(numero) is not int or not 1 <= numero <= len(intentos):
    print("Elegí un número de intento de la tabla.")
elif not ALUMNO.strip() or not COHORTE.strip() or not ENDPOINT.strip():
    print("Completá ALUMNO, COHORTE y ENDPOINT para enviar el intento.")
elif not re.fullmatch(
    r"https://script\.google\.com/(?:macros|a/macros/[A-Za-z0-9.-]+)/s/[A-Za-z0-9_-]+/exec",
    ENDPOINT.strip(),
):
    print("ENDPOINT debe ser la URL de Google Apps Script que termina en /exec.")
else:
    elegido = intentos[numero - 1]
    print(f"Intento {numero} · {elegido['modelo']} · Mejor MSE val. {min(elegido['val_mse']):.6f}")
    enviar_resultado(elegido, ALUMNO.strip(), COHORTE.strip(), ENDPOINT.strip(),
                     getpass("Token de la clase: ").strip())

### Preguntas para discutir

- Con el mismo presupuesto de épocas, ¿qué propuesta logró el menor MSE de validación?
- ¿Cuántos parámetros necesitó y cómo se compara con una alternativa más pequeña?
- ¿Qué cambió al agregar la derivada con el resto de la arquitectura fijo?